# working with icechunk locally

In [1]:
import icechunk
import tempfile
import zarr
import scanpy as sc
import pandas as pd
import numpy as np

Create icechunk repo

In [23]:
storage = icechunk.local_filesystem_storage(tempfile.TemporaryDirectory().name)
repo = icechunk.Repository.create(storage)

  2025-08-07T23:54:44.311814Z  WARN icechunk::storage::object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk/src/storage/object_store.rs:80



In [24]:
session = repo.writable_session("main")

In [25]:
store = session.store  # create a zarr store tracked with icechunk

In [26]:
group = zarr.group(store) #open the zarr group

Get scanpy 3k anndata and write to icechunk zarr store manually

In [27]:
adata = sc.read_h5ad('scanpy_3k.h5ad')

In [28]:
cell_types = pd.unique(adata.obs['cell_type']).tolist()

In [29]:
obs_group = group.create_group('obs')
sample = obs_group.create_group('None')
sample.create_array('categories', data=np.array(cell_types))
sample.create_array('codes', data=np.array([cell_types.index(c) for c in adata.obs['cell_type']]))

values = obs_group.create_group('values')
values.create_array(name='values', data=adata.obs['values'])

/home/workspace/environment/minimal/lib/python3.11/site-packages/zarr/core/dtype/npy/string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=11, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


<Array <icechunk.store.IcechunkStore object at 0x7b46a6f76410>/obs/values/values shape=(2700,) dtype=float64>

Commit the changes

In [30]:
snapshot_id_1 = session.commit("first commit")
print(snapshot_id_1)

FFWR8QFTW0C9JZ073N8G
